In [1]:
pip install dash plotly pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import plotly.express as px

from dash import Dash, html, dcc, Input, Output

In [3]:
spacex_df = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
)

In [4]:
app = Dash(__name__)
spacex_df = pd.read_csv("spacex_launch_dash.csv")
spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [5]:
# Task1: Add a Launch Site Drop-down Input Component

In [6]:
app.layout = html.Div(children=[

    # Task 1: Launch Site Drop-down Input Component
    html.H1(
        "SpaceX Launch Records Dashboard",
        style={'textAlign': 'center'}
    ),

    dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label': 'All Sites', 'value': 'ALL'},
        ] + [
            {'label': site, 'value': site}
            for site in sorted(spacex_df['Launch Site'].unique())
        ],
        value='ALL',
        placeholder='Select a Launch Site here',
        searchable=True
    ),

    html.Br(),

    # Task 2: Pie chart placeholder
    dcc.Graph(id='success-pie-chart'),

    html.Br(),

    # Task 3: Payload slider
    dcc.RangeSlider(
        id='payload-slider',
        min=0,
        max=10000,
        step=1000,
        marks={
            0: '0',
            1000: '1000',
            2000: '2000',
            3000: '3000',
            4000: '4000',
            5000: '5000',
            6000: '6000',
            7000: '7000',
            8000: '8000',
            9000: '9000',
            10000: '10000'
        },
        value=[
            spacex_df['Payload Mass (kg)'].min(),
            spacex_df['Payload Mass (kg)'].max()
        ]
    ),

    html.Br(),

    # Task 4: Scatter plot placeholder
    dcc.Graph(id='success-payload-scatter-chart')

])

In [7]:
# Task 2: Add a callback function to render success-pie-chart based on selected site dropdown

In [8]:
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):

    if entered_site == 'ALL':
        # Total successful launches for each launch site
        fig = px.pie(
            spacex_df.groupby('Launch Site')['class'].sum().reset_index(),
            values='class',
            names='Launch Site',
            title='Total Successful Launches by Site'
        )
        return fig

    else:
        # Filter data for the selected launch site
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]

        # Success vs Failure for the selected site
        fig = px.pie(
            filtered_df,
            names='class',
            title=f'Success vs Failure for {entered_site}'
        )
        return fig

In [9]:
# Task 3: Add a Range Slider to Select Payload

In [10]:
min_payload = spacex_df['Payload Mass (kg)'].min()
max_payload = spacex_df['Payload Mass (kg)'].max()

app.layout.children.extend([

    dcc.RangeSlider(
        id='payload-slider',
        min=0,
        max=10000,
        step=1000,
        marks={
            0: '0',
            1000: '1000',
            2000: '2000',
            3000: '3000',
            4000: '4000',
            5000: '5000',
            6000: '6000',
            7000: '7000',
            8000: '8000',
            9000: '9000',
            10000: '10000'
        },
        value=[min_payload, max_payload]
    ),

])

In [11]:
# Task 4: Add a callback function to render the success-payload-scatter-chart scatter plot

In [12]:
@app.callback(
    Output(component_id='success-payload-scatter-chart',
           component_property='figure'),
    [
        Input(component_id='site-dropdown',
              component_property='value'),
        Input(component_id='payload-slider',
              component_property='value')
    ]
)
def update_scatter_plot(selected_site, payload_range):

    low, high = payload_range

    # Filter by payload range
    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= low) &
        (spacex_df['Payload Mass (kg)'] <= high)
    ]

    if selected_site == 'ALL':

        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title='Payload vs. Launch Outcome for All Sites'
        )

    else:

        filtered_df = filtered_df[
            filtered_df['Launch Site'] == selected_site
        ]

        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title=f'Payload vs. Launch Outcome for {selected_site}'
        )

    return fig

In [13]:
if __name__ == "__main__":
    app.run(debug=True)

After visual analysis using the dashboard, you should be able to obtain some insights to answer the following five questions:

1) Which site has the largest successful launches?
2) Which site has the highest launch success rate?
3) Which payload range(s) has the highest launch success rate?
4) Which payload range(s) has the lowest launch success rate?
5) Which F9 Booster version (v1.0, v1.1, FT, B4, B5, etc.) has the highest
launch success rate?

1) CCAFS SLC-40 (Cape Canaveral Air Force Station Space Launch Complex 40)
It has the highest number of successful launches because it has the largest number of total launches in the dataset.

2) KSC LC-39A (Kennedy Space Center Launch Complex 39A)
It has the highest percentage of successful launches compared with the number of attempts.

3) Higher payload ranges generally have the highest success rates, especially:
4000–6000 kg
6000–8000 kg
8000–10000 kg
These ranges correspond mostly to later Falcon 9 missions using improved booster versions.

4) 0–2000 kg payload range
This range contains more early Falcon 9 missions, including early booster versions with more failures.

5) B5 (Block 5) has the highest success rate.
Other high-performing versions:
FT (Full Thrust) → very high success rate
B4 → high success rate